In [117]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import re
import json
from groq import Groq
import time
import csv
import psycopg2

In [ ]:
# model = SentenceTransformer('all-MiniLM-L6-v2')

# # Load the CSV file
# csv_file = 'chatbot-combined.csv'
# data = pd.read_csv(csv_file)

# # Generate embeddings for the questions
# data['embedding'] = data['question'].apply(lambda x: model.encode(x).tolist())


In [119]:
def get_db_connection():
    # Connect to your PostgreSQL database
    conn = psycopg2.connect(
        dbname="my_text_db",
        user="postgres",
        password="1234",
        host="localhost",  # Or the IP address of your PostgreSQL server
        port="5432"        # Default port for PostgreSQL
    )
    return conn


In [ ]:
# conn = get_db_connection()
# cur = conn.cursor()

# # Insert data into the PostgreSQL table
# for index, row in data.iterrows():
#     question = row['question']
#     answer = row['answer']
#     embedding = row['embedding']  # This is a list of floats
    
#     # Convert the embedding list to a format suitable for PostgreSQL
#     embedding_str = '[' + ','.join(map(str, embedding)) + ']'
    
#     insert_query = """
#     INSERT INTO questions (question, answer, embedding)
#     VALUES (%s, %s, %s::vector)
#     """
    
#     cur.execute(insert_query, (question, answer, embedding))

# # Commit changes and close the connection
# conn.commit()
# cur.close()
# conn.close()

# print("Data inserted successfully!")


Data inserted successfully!


In [120]:
def get_top_answer(query, model, conn, top_k=10):
    # Generate embedding for the input query
    query_embedding = model.encode(query, convert_to_tensor=False)
    query_embedding = np.array(query_embedding, dtype='float32')

    # Convert query_embedding to string format suitable for PostgreSQL
    embedding_str = '[' + ','.join(map(str, query_embedding)) + ']'

    cursor = conn.cursor()

    # Perform similarity search using pgvector's <=> operator for cosine similarity
    cursor.execute("""
    SELECT id, question, answer, embedding
    FROM questions
    ORDER BY embedding <=> %s
    LIMIT %s
    """, (embedding_str, top_k))

    rows = cursor.fetchall()

    retrieved_docs = []
    for row in rows:
        retrieved_docs.append(row[2])  # Answer column

    cursor.close()

    return retrieved_docs


In [121]:
# conn = get_db_connection()
# query = "What is the addmission process and scholarship criteria"
# result = get_top_answer(query,model,conn)
# conn.close()

In [122]:
grok_api_key = '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'

In [123]:
def GroqChat(question):
    client = Groq(
        api_key=grok_api_key,

    )

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": question,
            }
        ],
            model = "llama-3.1-70b-versatile"
    )

    cleaned_json_string = chat_completion.choices[0].message.content

    json_str = re.sub(r'}\s*{', '}, {', cleaned_json_string)
    return json_str

In [ ]:
def generate_answer_from_docs(query, retrieved_docs):
    result = []
    if not retrieved_docs:
        return "Don't have an answer for the query."

    context = "\n".join(retrieved_docs)
    result.append(context)

    prompt = f"Answer the following query, based only on the given context. Do not add anything from your previous learnings. Do not state in answer that a context is provided to you. If the context seems irrelevant just say 'I don't have an appropriate answer'. query: {query} context: {context}"
    groq_answer = GroqChat(prompt)
    result.append(groq_answer)
    result.append('')
    return result


In [ ]:
if __name__ == "__main__":
    
    model = SentenceTransformer('all-MiniLM-L6-v2')

    
    # Establish DB connection
    conn = get_db_connection()



    while True:
        query = input("Enter your query here: ")
        if query.lower() == "exit":
            break

        retrieved_docs = list(set(get_top_answer(query, model, conn)))
        generated_answer = generate_answer_from_docs(query, retrieved_docs)
        print('You: ',query)
        print("Llama Answer: ", generated_answer[1])
        
        print()

    conn.close()  # Close DB connection when done


You:  co founder of S U?
Llama Answer:  Co-founder is not explicitly mentioned, but based on the text it seems that the founder is Dr. Amit Singhal, and a crucial figure alongside him is Dr. Anuja Agarwal, referred to as the "founding Dean" of Sitare University.
["The university is being completely funded by its founder, Dr. Amit Singhal.\nDr. Amit Singhal was one of the early employees of Google, and worked there for more than fifteen years. He was in-charge of Google Search for a very long time. He benefitted from Google's financial success, becoming quite financially successful themselves. He wanted to use majority of their net worth to do something that would transform society, and after a lot of brainstorming, he settled down on educating bright students from economically weak backgrounds. Dr. Amit has not only put in his own money into this initiative, he is also investing most of his time and energy in making sure that Sitare University turns out to be a global leader in Compute